In [1]:
import numpy as np
import pandas as pd
import torch
#import wandb
import random
#from joblib import dump
import itertools
import torch
from torch import nn
from torch.nn import functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split
from tqdm.auto import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EvalPrediction
)
#from datasets import Dataset
#from constants import *
import os

/home/skunk/routing2/lib/python3.10/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/home/skunk/routing2/lib/python3.10/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


# Data

In [2]:
data = pd.read_pickle('dataset/routerbench_0shot.pkl')
inputs = np.load('dataset/OAIEmbeddings_routerbench.npy')
data = data[[not s.lower().startswith('chinese') for s in data['eval_name'].values]] # remove the chinese questions
gpt_4_idx = 5
mixtral_idx = 9
idxs= [gpt_4_idx, mixtral_idx]
labels = (data.values[:,idxs]).astype('float')
pref_labels = []
four_way_labels = []
for label in labels:
    if int(label[0])==1 and int(label[1])==0: pref_labels.append([1.0])
    else: pref_labels.append([0.0])
for label in labels:
    if int(label[0])==1 and int(label[1])==0: four_way_labels.append([0.0])
    elif int(label[0])==0 and int(label[1])==0: four_way_labels.append([1.0])
    elif int(label[0])==1 and int(label[1])==1: four_way_labels.append([2.0])
    else: four_way_labels.append([3.0])
seed = 4

# Not Diamond

In [3]:
train_idx, test_idx, train_inputs,\
    test_inputs, train_labels,\
    test_labels = train_test_split(np.arange(len(inputs)), \
                                   inputs, four_way_labels, train_size=0.8, random_state=seed)

In [4]:
rf_classifier = RandomForestClassifier(
            max_depth= 20,
            max_features=1.0,
            n_estimators=100,
            n_jobs=-1,
            random_state=4
        )
rf_classifier.fit(train_inputs, train_labels)

/home/skunk/routing2/lib/python3.10/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestClassifier(max_depth=20, max_features=1.0, n_jobs=-1,
                       random_state=4)

In [5]:
probs = rf_classifier.predict_proba(inputs)

In [6]:
gpt_prob = probs[:, 0] +probs [:, 1]

In [7]:
data_path = 'dataset'
data_sms = pd.read_pickle('dataset/sms_roberta_base_v3.pkl')
data_sms['NotDiamond_probs'] = gpt_prob
data_sms.to_pickle('dataset/sms_roberta_base_v3.pkl')

# MF Model

In [8]:
train_idx, test_idx, train_inputs,\
    test_inputs, train_labels,\
    test_labels = train_test_split(np.arange(len(inputs)), \
                                   inputs, pref_labels, train_size=0.8, random_state=seed)

In [9]:
class MFModel_Train(torch.nn.Module): #adapted from https://github.com/lm-sys/RouteLLM/blob/main/routellm/routers/matrix_factorization/train_matrix_factorization.py
    def __init__(
        self,
        embeddings,
        dim=128
    ):
        super().__init__()
        num_prompts,text_dim = embeddings.shape
        num_classes=1
        use_proj=True
        
        self.use_proj = use_proj
        self.P = torch.nn.Embedding(1, dim)
        self.Q = torch.nn.Embedding(num_prompts, text_dim).requires_grad_(False) 
        self.Q.weight.data.copy_(embeddings)

        if self.use_proj:
            self.text_proj = torch.nn.Linear(text_dim, dim, bias=False)
        else:
            assert (
                text_dim == dim
            ), f"text_dim {text_dim} must be equal to dim {dim} if not using projection"

        self.classifier = nn.Linear(
            dim, num_classes, bias=False
        )  # bias should be False!

    def get_device(self):
        return self.P.weight.device

    def forward(self, prompt, test=False, alpha=0.05):
        prompt = prompt.to(self.get_device())
        model_embed = self.P(torch.tensor(0).to(self.get_device()))[None,:]
        model_embed = F.normalize(model_embed, p=2, dim=1)
        prompt_embed = self.Q(prompt)
        if not test:
            # adding noise to stablize the training
            prompt_embed += torch.randn_like(prompt_embed) * alpha
        if self.use_proj:
            prompt_embed = self.text_proj(prompt_embed)

        return self.classifier(
            model_embed * prompt_embed
        ).squeeze()

    @torch.no_grad()
    def predict_proba(self, embedding):
        model_embed = self.P(torch.tensor(0).to(self.get_device()))[None,:]
        model_embed = F.normalize(model_embed, p=2, dim=1)
        prompt_embed = torch.tensor(embedding, dtype=torch.float32).to(self.get_device())

        if self.use_proj:
            prompt_embed = self.text_proj(prompt_embed)

        return self.classifier(
            model_embed * prompt_embed
        ).squeeze().detach().cpu().numpy()

def train_mf_model(embeddings,
                   Y,
                   epochs=5,
                   batch_size=64,
                   lr=3e-4,
                   weight_decay=1e-5,
                   val_split=0.2,
                   validate_every=10,  # Validate every 'm' steps
                   device='cuda' if torch.cuda.is_available() else 'cpu'):
    
    # Convert inputs to tensors if they aren't already
    embeddings = torch.tensor(embeddings, dtype=torch.float32).to(device)
    Y = torch.tensor(Y, dtype=torch.float32).to(device)
    
    # Create dataset and split into train/val
    dataset = TensorDataset(torch.arange(len(embeddings)), Y)
    val_size = int(len(dataset) * val_split)
    train_size = len(dataset) - val_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    
    # Initialize model and move to device
    model = MFModel_Train(embeddings).to(device)
    
    # Initialize optimizer and loss function
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()
    
    # Keep track of best validation accuracy and corresponding model weights
    best_val_acc = 0.0
    best_model_weights = None
    
    step = 0  # Global step counter
    for epoch in tqdm(range(epochs)):
        # Training phase
        model.train()
        train_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for indices, labels in train_loader:
            indices, labels = indices.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(indices)
            loss = criterion(outputs.unsqueeze(1), labels)
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            correct_train += (predicted == labels).sum().item()
            total_train += labels.size(0)
            
            # Increment step counter
            step += 1
            
            # Validation every 'validate_every' steps
            if step % validate_every == 0:
                model.eval()
                correct_val = 0
                total_val = 0
                
                with torch.no_grad():
                    for val_indices, val_labels in val_loader:
                        val_indices, val_labels = val_indices.to(device), val_labels.to(device)
                        
                        val_outputs = model(val_indices, test=True)  # Use test=True for validation
                        val_predicted = (torch.sigmoid(val_outputs) > 0.5).float()
                        correct_val += (val_predicted == val_labels).sum().item()
                        total_val += val_labels.size(0)
                
                val_acc = 100 * correct_val / total_val
                
                # Save best model
                if val_acc > best_val_acc:
                    best_val_acc = val_acc
                    best_model_weights = model.state_dict().copy()
                
                #print(f'Step [{step}] - Val Acc: {val_acc:.2f}%')
                model.train()
        
        # Calculate epoch metrics
        train_acc = 100 * correct_train / total_train
    
    # Load best model weights
    model.load_state_dict(best_model_weights)
    #print(f'Best Validation Accuracy: {best_val_acc:.2f}%')
    
    return model.to('cpu')


# Train and Predict

In [10]:
MF_model = train_mf_model(train_inputs, train_labels)

  0%|          | 0/5 [00:00<?, ?it/s]

In [11]:
def sigmoid(z):
    return 1/(1+np.exp(-z))
MF_model.eval()
pred = sigmoid(MF_model.predict_proba(inputs))

In [12]:
data_path = 'dataset'
data_sms = pd.read_pickle('dataset/sms_roberta_base_v3.pkl')
data_sms['LLMbench_probs'] = pred
data_sms.to_pickle('dataset/sms_roberta_base_v3.pkl')